# 03. PDF/HWP 텍스트 추출 검증
원본 RFP 파일을 직접 읽고 PDF/HWP 추출 성공 여부와 실패 사유를 확인합니다.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebook':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.parser_chunker import read_document

with (PROJECT_ROOT / 'config/default.yaml').open(encoding='utf-8') as f:
    config = yaml.safe_load(f)

metadata_path = Path(config['paths']['metadata'])
raw_documents_path = Path(config['paths']['raw_documents'])
failure_log_path = PROJECT_ROOT / config['paths']['extraction_failures']

metadata = pd.read_csv(metadata_path)
print(f'메타데이터: {len(metadata)}건')
print(f'원본 폴더: {raw_documents_path}')

메타데이터: 100건
원본 폴더: /data/original_data/files


In [2]:
extraction_rows = []

for index, row in metadata.iterrows():
    doc_id = f'doc_{index + 1:03d}'
    file_name = str(row['파일명']).strip()
    file_path = raw_documents_path / file_name
    file_type = file_path.suffix.lower().lstrip('.')

    try:
        text = read_document(file_path)
        if text:
            status = 'success'
            error = ''
        else:
            status = 'empty'
            error = '추출된 텍스트가 비어 있습니다.'
    except Exception as exc:
        text = ''
        status = 'error'
        error = f'{type(exc).__name__}: {exc}'

    extraction_rows.append({
        'doc_id': doc_id,
        'file_name': file_name,
        'file_type': file_type,
        'status': status,
        'text_length': len(text),
        'error': error,
        'text_preview': text[:200].replace('\n', ' '),
    })

extraction_result = pd.DataFrame(extraction_rows)
display(extraction_result.groupby(['file_type', 'status']).size().rename('count').reset_index())

,file_type,status,count
0,hwp,success,96
1,pdf,success,4


In [3]:
successful_samples = (
    extraction_result[extraction_result['status'] == 'success']
    .groupby('file_type', as_index=False)
    .first()
)
display(successful_samples[['doc_id', 'file_name', 'file_type', 'text_length', 'text_preview']])

failures = extraction_result[extraction_result['status'] != 'success'].copy()
failure_log_path.parent.mkdir(parents=True, exist_ok=True)
failures[['doc_id', 'file_name', 'file_type', 'status', 'error']].to_csv(
    failure_log_path, index=False, encoding='utf-8-sig'
)

print(f'추출 성공: {(extraction_result["status"] == "success").sum()}건')
print(f'빈 텍스트/오류: {len(failures)}건')
print(f'실패 로그: {failure_log_path}')
if not failures.empty:
    display(failures[['doc_id', 'file_name', 'file_type', 'status', 'error']])

,doc_id,file_name,file_type,text_length,text_preview
0,doc_001,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,hwp,42424,2024년 특성화 맞춤형 교육환경 구축 – 트랙운영 학사정보시스템 고도화 제안요청서...
1,doc_008,고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf,pdf,230223,-1- 제 안 요 청 서 고려대학교 차세대 포털·학사 정보...


추출 성공: 100건
빈 텍스트/오류: 0건
실패 로그: /home/jaeyeol/sprint-ai-mid-project_team3/data/processed/extraction_failures.csv
